In [ ]:
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import LineString
from src.smooth_trajectory import smooth_distances_per_trip

from src.constants import (
    CA_NAD83_Albers,
    CULVER_CITY_FEED_KEY,
    MAX_SNAP_DISTANCE_M,
    SHAPE_KEY_TO_SHAPE_ID_MAP,
)
from src._data_loaders import (
    get_matched_vehicle_positions,
    get_selected_shapes,
    get_smoothed_vehicle_positions,
    get_traffic_signals,
)
from src.match_shapes_vp import project_points_on_shape
from src.delay_calculation import estimate_intersection_time_single_trip
from src.plot_trip import plot_distance_over_time, plot_speed_over_time


shape_ids = list(SHAPE_KEY_TO_SHAPE_ID_MAP.values())

In [ ]:
# for caltrans network only

import os

os.environ["REQUESTS_CA_BUNDLE"] = r"C:\Users\s163107\Documents\CTROOTCA01.cer"

In [ ]:
# Select the trip to analyze: a TRIP_KEY on a SERVICE_DATE.
SERVICE_DATE = "2026-02-04"
TRIP_KEY = "986.0"  # e.g. "1026.0"; None plots the first available trip for the date

In [ ]:
vp = get_matched_vehicle_positions(SERVICE_DATE)

In [ ]:
signals = get_traffic_signals()
stops = gpd.read_file("data/stops_shp-1-05.geojson").to_crs(CA_NAD83_Albers)

In [ ]:
trip = vp.loc[vp["TRIP_KEY"] == TRIP_KEY]

In [ ]:
# outlier removal
absolute_distance_since_last_ping = abs(trip["distance_along_shape"] - trip["distance_along_shape"].shift(1))
trip_filtered = trip.loc[absolute_distance_since_last_ping < 200]

In [ ]:
trip

In [ ]:
calculated_smoothed = smooth_distances_per_trip(trip_filtered, trip_filtered["distance_along_shape"], 1)
calculated_smoothed

In [ ]:
plot_distance_over_time(
    trips=[trip_filtered],
    signal_distances=pd.Series(),
    nearside_stop_distances=pd.Series(),
    farside_stop_distances=pd.Series(),
    signal_delays_s=pd.Series(),
    trips_smoothed=[calculated_smoothed],
    highlight_delay_threshold_s=0,
)

In [ ]:
cached_smoothed = get_smoothed_vehicle_positions(service_date=SERVICE_DATE)
cached_smoothed_selected_trip = cached_smoothed.loc[cached_smoothed["TRIP_KEY"] == TRIP_KEY]
cached_smoothed_selected_trip

In [ ]:
plot_distance_over_time(
    trips=[trip],
    signal_distances=pd.Series(),
    nearside_stop_distances=pd.Series(),
    farside_stop_distances=pd.Series(),
    signal_delays_s=pd.Series(),
    trips_smoothed=[cached_smoothed_selected_trip],
    highlight_delay_threshold_s=0,
)